In [1]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
Koopman Closed-Loop — IMPLEMENTACIÓN PLANTA FÍSICA REAL
========================================================
Adaptado para:
  - Tanques cilíndricos pequeños (Atank = 3.02e-3 m²)
  - Bombas limitadas al 70% (qi1max=2.69e-5, qi3max=2.41e-5 m³/s)
  - Sensores ultrasónicos HC-SR04 en tanques 1 y 3 (h2 NO medido)
  - ESP32 COM6, protocolo serial 115200 baud
  - Referencia: 0.10 → 0.20 → 0.15 m  (bajada limitada a ~37.4 cm/s)
  - Observador Luenberger + KoopmanMPC (N=20)

INSTALACIÓN PREVIA:
  pip install numpy scipy casadi pyserial matplotlib
"""

import numpy as np
from scipy.optimize import least_squares
import scipy.linalg as la
import casadi as ca
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
import time
import serial
import serial.tools.list_ports
import threading
import queue
import sys
from datetime import datetime

warnings.filterwarnings('ignore')

output_dir = Path('./outputs_real')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS — CALIBRADOS DESDE MEDICIÓN REAL
# ============================================================================
# Calibración de alphao (Torricelli):
#   Vaciado tanque 1: h=0.15m → h=0.05m en t=105s, bombas apagadas
#   Integración analítica:
#     alphao * Ao = Atank * 2*(sqrt(0.15)-sqrt(0.05)) / (sqrt(2*g) * 105)
#     alphao * Ao = 3.02e-3 * 0.3274 / (4.429 * 105) = 2.13e-6 m^2.5/s
#   Con Do=10mm → Ao = pi*(0.01)^2/4 = 7.85e-5 m²
#     alphao = 2.13e-6 / 7.85e-5 = 0.0271
#   Se asume mismo alphao para T1, T2, T3 (mismos orificios de fondo)
#
# Conexiones inter-tanques: Do=10mm (medido), A=7.85e-5 m²
#   alpha120/alpha230 se conservan del ajuste original (flujo viscoso)
#
P_NOM = {
    # --- Geometría de los tanques (cilíndricos, ø=6.2 cm) ---
    'Atank'   : 3.02e-3,     # m²  — π*(0.062/2)²

    # --- Fluidos ---
    'rho'     : 997.0,       # kg/m³
    'eta'     : 8.9e-4,      # Pa·s
    'g'       : 9.81,        # m/s²

    # --- Orificios de salida — CALIBRADOS desde vaciado 15→5 cm en 105 s ---
    #     alphao = 0.0271, Do = 10 mm → Ao = 7.85e-5 m²
    'alphao1' : 0.0271, 'Do1': 10e-3,   # orificio tanque 1
    'alphao2' : 0.0271, 'A2' : 7.85e-5, # orificio tanque 2 (mismo tipo)
    'alphao3' : 0.0271, 'Do3': 10e-3,   # orificio tanque 3

    # --- Conexiones inter-tanques — Do=10mm medido ---
    'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,

    # --- Caudales máximos bombas (ya limitadas al 70%) ---
    'qi1max'  : 2.69e-5,     # m³/s — bomba 1 al 100% de señal → 70% real
    'qi3max'  : 2.41e-5,     # m³/s — bomba 3 al 100% de señal → 70% real
}

# ============================================================================
# PARÁMETROS DE SIMULACIÓN / CONTROL
# ============================================================================
Ts        = 2.0        # s — periodo de muestreo (igual que ESP32)
T_sim     = 800        # s — duración total del experimento
Nsim      = int(T_sim / Ts)
sigma     = 1e-3       # m — ruido de medición (1 mm, ajusta si es ruidoso)
EPS       = 1e-10
EPS_PSI   = 1e-8
SETTLING_S = 60        # s — ventana de transitorio excluida en métricas

# Puerto serial
SERIAL_PORT = 'COM6'
BAUD_RATE   = 115200
SERIAL_TIMEOUT = 3.0   # s

# Secuencia de referencias (en metros)
# 0.10 → 0.20 → 0.15 m  (la bajada 0.20→0.15 está calibrada para que las bombas aguanten)
REF_STEPS = [
    (0,           0.10),   # 0..266s   → h2 = 10 cm
    (int(Nsim*0.33), 0.20),  # 267..532s → h2 = 20 cm
    (int(Nsim*0.66), 0.15),  # 533..800s → h2 = 15 cm
]

# ============================================================================
# MODELO INTERNO (para EDMD y predictor Koopman)
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                   * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x + dt/2*k1, u, p)
    k3 = nl_ode(x + dt/2*k2, u, p); k4 = nl_ode(x + dt*k3, u, p)
    return np.clip(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0, 0.27)  # antes 0.25

def psi(x):
    """8 observables de Koopman."""
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1 * h2, h2 * h3, h1 * h3,
        h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, p=P_NOM):
    """
    Punto de operación estacionario para h2=h2_ref.
    Permite u3>0 (regularizado para preferir u3≈0 cuando sea posible),
    ya que para h2=0.20m puede no existir SS factible con u3=0 dentro
    del rango físico [0,0.25]m.
    """
    w_reg = 0.05  # penaliza u3 para preferir soluciones con u3≈0 si existen
    def eqs(v):
        h1, h3, u1, u3 = v
        f = nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                   [np.clip(u1, 0, 1), np.clip(u3, 0, 1)], p)
        return np.concatenate([f, [w_reg * u3]])

    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5, 0.0),
        (h2_ref*1.6, h2_ref*0.7, 0.7, 0.0),
        (h2_ref*2.0, h2_ref*0.8, 0.9, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.7, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.75, 0.0),
        (h2_ref*1.8, h2_ref*0.85, 0.85, 0.0),
        (h2_ref*1.3, h2_ref*0.9, 1.0, 0.1),
        (h2_ref*1.2, h2_ref*1.0, 1.0, 0.3),
        (h2_ref*1.1, h2_ref*1.1, 1.0, 0.5),
        (h2_ref*1.0, h2_ref*1.2, 1.0, 0.8),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0, 0.0],
                                      [0.27, 0.27, 1.0, 1.0]),
                              max_nfev=5000)
            phys_res = np.linalg.norm(s.fun[:3])  # solo las 3 ecuaciones físicas
            if s.success and phys_res < best_res:
                best_res = phys_res; best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return (np.array([best[0], h2_ref, best[1]]),
                np.array([np.clip(best[2], 0, 1), np.clip(best[3], 0, 1)]))
    return None, None

# ============================================================================
# COMUNICACIÓN SERIAL CON ESP32
# ============================================================================
class ESP32Interface:
    """
    Maneja la comunicación serial con el ESP32 en un hilo separado.
    - Lectura asíncrona de mediciones (h1, h2_real, h3)
    - Escritura de comandos (u1, u2)
    """
    def __init__(self, port=SERIAL_PORT, baud=BAUD_RATE):
        self.port = port
        self.baud = baud
        self.ser  = None
        self.rx_queue = queue.Queue(maxsize=20)
        self._stop_event = threading.Event()
        self._last_meas  = None
        self._lock       = threading.Lock()

    def connect(self):
        print(f"\nConectando a {self.port} @ {self.baud} baud...")
        try:
            self.ser = serial.Serial(
                self.port, self.baud,
                timeout=SERIAL_TIMEOUT,
                write_timeout=1.0
            )
            time.sleep(2.0)   # espera reset del ESP32
            self.ser.reset_input_buffer()
            print(f"  ✓ Conectado a {self.port}")
        except serial.SerialException as e:
            print(f"  ✗ Error: {e}")
            print("\nPuertos disponibles:")
            for p in serial.tools.list_ports.comports():
                print(f"    {p.device} — {p.description}")
            raise

        # Hilo lector
        self._reader_thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._reader_thread.start()

    def _reader_loop(self):
        """Lee líneas del ESP32 y las pone en la cola."""
        while not self._stop_event.is_set():
            try:
                if self.ser.in_waiting:
                    line = self.ser.readline().decode('utf-8', errors='ignore').strip()
                    if line:
                        self.rx_queue.put_nowait(line)
            except Exception:
                pass
            time.sleep(0.005)

    def get_measurement(self, timeout=3.0):
        """
        Espera y devuelve (h1, h2_real, h3) en metros desde el ESP32.
        Formato esperado: "0.0218,0.0157,0.0235"
        h2_real es la verdad terreno de h2 (medida en la planta física),
        NO se usa para corregir el observador Luenberger (que solo mide
        h1 y h3) — solo sirve para validar la estimación de h2 contra
        el valor real.
        """
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                line = self.rx_queue.get(timeout=0.1)
                parts = line.split(',')
                if len(parts) == 3:
                    h1      = float(parts[0])
                    h2_real = float(parts[1])
                    h3      = float(parts[2])
                    h1      = max(h1, 0.0)
                    h2_real = max(h2_real, 0.0)
                    h3      = max(h3, 0.0)
                    with self._lock:
                        self._last_meas = (h1, h2_real, h3)
                    return h1, h2_real, h3
            except queue.Empty:
                pass
        # Si hay timeout, devolver última medición conocida
        with self._lock:
            if self._last_meas is not None:
                print("  ⚠ Timeout serial — usando última medición")
                return self._last_meas
        raise TimeoutError("No se recibieron datos del ESP32")

    def send_control(self, u1: float, u2: float):
        """Envía "u1,u2\n" al ESP32."""
        u1 = float(np.clip(u1, 0.0, 1.0))
        u2 = float(np.clip(u2, 0.0, 1.0))
        cmd = f"{u1:.4f},{u2:.4f}\n"
        try:
            self.ser.write(cmd.encode('utf-8'))
        except Exception as e:
            print(f"  ✗ Error al enviar: {e}")

    def stop_pumps(self):
        """Apaga las bombas de forma segura."""
        self.send_control(0.0, 0.0)
        print("  Bombas apagadas.")

    def close(self):
        self._stop_event.set()
        self.stop_pumps()
        time.sleep(0.5)
        if self.ser and self.ser.is_open:
            self.ser.close()
        print("  Puerto serial cerrado.")

# ============================================================================
# ENTRENAMIENTO EDMD
# ============================================================================
def train_edmd(verbose=True):
    if verbose:
        print("\nEntrenando EDMD (parámetros nominales)...")

    rng = np.random.default_rng(42)
    z_k, u_k, z_k1 = [], [], []

    # Bloque 1: perturbaciones normales alrededor de los SS
    for h2r in [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None:
            continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(170):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35*err, -0.35, 0.35) + rng.uniform(-0.15, 0.15)
                u[1] += rng.uniform(-0.05, 0.05); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    # Bloque 2: perturbaciones grandes ±40%
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None:
            continue
        for pert in [0.60, 0.70, 0.75, 1.25, 1.35]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6*err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    # Bloque 3: tránsitos entre referencias
    for (h2f, h2t) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None:
            continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2t; u = uf.copy()
            u[0] += np.clip(-0.5*err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4(x, u); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    Z = np.array(z_k); U = np.array(u_k); Zn = np.array(z_k1)
    nz, nu = Z.shape[1], U.shape[1]
    if verbose:
        print(f"  {len(Z)} muestras de entrenamiento")

    # Regresión EDMD con Tikhonov
    Phi = np.hstack([Z, U]); lam = 1e-4
    Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
    A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T

    # Estabilización espectral
    vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
    vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
    A_d = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))

    rmse = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
    eigs = np.abs(np.linalg.eigvals(A_d))
    if verbose:
        print(f"  RMSE one-step: {rmse:.2e}")
        print(f"  |λ| ∈ [{eigs.min():.5f}, {eigs.max():.5f}] ✓")
    return A_d, B_d, nz, nu

def design_observer(A_d, nz):
    """Diseña ganancia Luenberger por DARE."""
    # C_obs mide √h1, √h3, h1, h3 (filas 0,2,6,7 del vector observable)
    C_obs = np.zeros((4, nz))
    C_obs[0, 0] = 1.0   # √h1
    C_obs[1, 2] = 1.0   # √h3
    C_obs[2, 6] = 1.0   # h1
    C_obs[3, 7] = 1.0   # h3
    m = C_obs.shape[0]

    print("\nDiseñando ganancia L (Luenberger)...")
    for alpha in [1.0, 10.0, 50.0, 100.0, 500.0, 1000.0]:
        try:
            Q_obs = alpha * np.eye(nz)
            X = la.solve_discrete_are(A_d.T, C_obs.T, Q_obs, np.eye(m))
            L_c = np.linalg.solve(C_obs @ X @ C_obs.T + np.eye(m), C_obs @ X @ A_d.T).T
            eigs_L = np.abs(np.linalg.eigvals(A_d - L_c @ C_obs)).max()
            if eigs_L < 0.999 and np.abs(L_c).max() < 500:
                print(f"  alpha={alpha}: max|λ|={eigs_L:.5f} ✓")
                return L_c, C_obs
        except:
            pass
    raise RuntimeError("No se pudo diseñar L. Revisa los parámetros del modelo.")

def build_mpc(A_d, B_d, nz, nu):
    """Construye el optimizador KoopmanMPC con CasADi/IPOPT."""
    print("\nConstruyendo KoopmanMPC...")
    N_mpc = 20; q_mpc = 500
    R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])

    Z_sym = ca.SX.sym('Z', nz, N_mpc+1)
    U_sym = ca.SX.sym('U', nu, N_mpc)
    z0_sym = ca.SX.sym('z0', nz)
    yr_sym = ca.SX.sym('yr')
    up_sym = ca.SX.sym('up', nu)

    A_cas = ca.DM(A_d); B_cas = ca.DM(B_d)
    J = 0; g = []; lbg = []; ubg = []

    g += [Z_sym[:, 0] - z0_sym]; lbg += [0.0]*nz; ubg += [0.0]*nz
    for k in range(N_mpc):
        g += [Z_sym[:, k+1] - A_cas@Z_sym[:, k] - B_cas@U_sym[:, k]]
        lbg += [0.0]*nz; ubg += [0.0]*nz
        h2_k = Z_sym[1, k+1]**2; ey = h2_k - yr_sym
        du = U_sym[:, 0] - up_sym if k == 0 else U_sym[:, k] - U_sym[:, k-1]
        wk = 10 if k == N_mpc-1 else 1
        J += (wk * q_mpc * ey**2
              + ca.mtimes(U_sym[:, k].T, ca.DM(R1) @ U_sym[:, k])
              + ca.mtimes(du.T,          ca.DM(R2) @ du))

    w_sym  = ca.vertcat(ca.reshape(Z_sym, nz*(N_mpc+1), 1), ca.reshape(U_sym, nu*N_mpc, 1))
    par_sym = ca.vertcat(z0_sym, yr_sym, up_sym)
    lbw = ([-ca.inf]*(nz*(N_mpc+1)) + list(np.tile(umin, N_mpc)))
    ubw = ([ ca.inf]*(nz*(N_mpc+1)) + list(np.tile(umax, N_mpc)))

    nlp = {'x': w_sym, 'f': J, 'g': ca.vertcat(*g), 'p': par_sym}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
            'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    solver = ca.nlpsol('solver_kmpc', 'ipopt', nlp, opts)
    print("  KoopmanMPC ✓")
    return solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz, nu

# ============================================================================
# LOOP DE CONTROL EN TIEMPO REAL
# ============================================================================
def run_real_experiment(esp: ESP32Interface,
                        A_d, B_d, L_d, C_obs,
                        solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg,
                        nz, nu, SS_NOM):
    """
    Bucle de control en tiempo real.
    En cada iteración:
      1. Espera la medición del ESP32 (sincronizado a Ts=2s)
      2. Corre el observador Koopman
      3. Resuelve KoopmanMPC
      4. Envía u al ESP32
    """

    # Secuencia de referencias
    yref_v = np.zeros(Nsim)
    for (k_start, ref_val) in REF_STEPS:
        yref_v[k_start:] = ref_val
    ref_change_times_s = [k * Ts for k, _ in REF_STEPS[1:]]

    # Estado inicial desde los sensores
    print("\nEsperando primera medición del ESP32...")
    h1_meas, h2_real, h3_meas = esp.get_measurement()
    print(f"  Primera medición: h1={h1_meas*100:.2f}cm | h2_real={h2_real*100:.2f}cm | h3={h3_meas*100:.2f}cm")

    # Inicializar observador SIN usar h2_real (h2 no se considera medido):
    # se parte de la referencia inicial como estimación cruda de arranque,
    # igual que antes. h2_real solo se usa para comparar, nunca para
    # inicializar ni corregir el observador.
    x_init = np.array([h1_meas, yref_v[0], h3_meas])
    z_obs  = psi(x_init)
    uprev  = np.array([0.0, 0.0])
    w0_mpc = None
    yr_prev = yref_v[0]

    # Historial
    T_hist  = np.arange(Nsim + 1) * Ts
    H_meas  = np.zeros((2, Nsim + 1))   # h1, h3 medidos
    H2_REAL = np.zeros(Nsim + 1)        # h2 verdad terreno (NO usado por el observador)
    X_hat   = np.zeros((3, Nsim + 1))   # estimación [h1, h2, h3]
    U_hist  = np.zeros((2, Nsim))
    step_ms = []

    H_meas[0, 0] = h1_meas; H_meas[1, 0] = h3_meas
    H2_REAL[0]   = h2_real
    X_hat[:, 0]  = x_init

    print(f"\n{'='*65}")
    print(f"  EXPERIMENTO INICIADO — {datetime.now().strftime('%H:%M:%S')}")
    print(f"  Referencia inicial: h2 = {yref_v[0]*100:.0f} cm")
    print(f"{'='*65}")
    print(f"  {'t[s]':>6} | {'h1[cm]':>7} | {'ĥ2[cm]':>7} | {'h2r[cm]':>7} | "
          f"{'eobs[cm]':>8} | {'h3[cm]':>7} | {'ref[cm]':>7} | {'u1':>5} | {'u3':>5} | {'ms':>5}")
    print(f"  {'-'*68}")

    t_experiment_start = time.time()

    for k in range(Nsim):
        t_step_start = time.time()
        yr = yref_v[k]

        # ── Warm-start al cambio de referencia ──────────────────────────
        if yr != yr_prev:
            w0_mpc = None
            print(f"\n  ▶ Cambio de referencia: {yr_prev*100:.0f} → {yr*100:.0f} cm")
        yr_prev = yr

        # ── 1. Predicción Koopman ─────────────────────────────────────
        z_pred = A_d @ z_obs + B_d @ uprev
        z_pred[:3] = np.maximum(z_pred[:3], 0)
        z_pred[6:] = np.maximum(z_pred[6:], 0)

        # ── 2. Esperar medición real del ESP32 (incluye h2 verdad terreno) ──
        # El ESP32 envía cada 2 s; nos sincronizamos a su ritmo
        h1_meas, h2_real, h3_meas = esp.get_measurement(timeout=Ts * 2)
        H_meas[0, k+1] = h1_meas; H_meas[1, k+1] = h3_meas
        H2_REAL[k+1]   = h2_real

        # ── 3. Corrección Luenberger ─────────────────────────────────
        y_obs = np.array([
            np.sqrt(max(h1_meas, EPS_PSI)),
            np.sqrt(max(h3_meas, EPS_PSI)),
            h1_meas,
            h3_meas,
        ])
        innov  = y_obs - C_obs @ z_pred
        z_obs  = z_pred + L_d @ innov
        z_obs[:3] = np.maximum(z_obs[:3], 0)
        z_obs[6:] = np.maximum(z_obs[6:], 0)

        # Reconstruir estado físico estimado
        x_hat = np.clip(z_obs[:3]**2, 0, 0.75)
        X_hat[:, k+1] = x_hat

        # ── 4. KoopmanMPC ────────────────────────────────────────────
        if w0_mpc is None:
            w0_mpc = np.zeros(nz*(N_mpc+1) + nu*N_mpc)
            w0_mpc[:nz] = z_obs
        par_val = np.concatenate([z_obs, [yr], uprev])
        try:
            sol   = solver(x0=w0_mpc, lbx=lbw, ubx=ubw,
                           lbg=lbg, ubg=ubg, p=par_val)
            w_opt = np.array(sol['x']).flatten()
            Z_opt = w_opt[:nz*(N_mpc+1)].reshape(N_mpc+1, nz)
            U_opt = w_opt[nz*(N_mpc+1):].reshape(N_mpc, nu)
            u_k   = np.clip(U_opt[0], umin, umax)
            # Warm-start desplazado
            Z_sh = np.vstack([Z_opt[1:], Z_opt[-1:]])
            U_sh = np.vstack([U_opt[1:], U_opt[-1:]])
            w0_mpc = np.concatenate([Z_sh.flatten(), U_sh.flatten()])
        except Exception as e:
            print(f"  ⚠ MPC falló en k={k}: {e} — usando u anterior")
            u_k = uprev.copy(); w0_mpc = None

        elapsed_ms = (time.time() - t_step_start) * 1000
        step_ms.append(elapsed_ms)
        U_hist[:, k] = u_k; uprev = u_k.copy()

        # ── 5. Enviar control al ESP32 ───────────────────────────────
        esp.send_control(u_k[0], u_k[1])

        # ── Log en consola ───────────────────────────────────────────
        if k % 5 == 0:
            e_obs_cm = (x_hat[1] - h2_real) * 100
            print(f"  {k*Ts:>6.0f} | {h1_meas*100:>7.2f} | {x_hat[1]*100:>7.2f} | "
                  f"{h2_real*100:>7.2f} | {e_obs_cm:>+8.3f} | {h3_meas*100:>7.2f} | {yr*100:>7.2f} | "
                  f"{u_k[0]:>5.3f} | {u_k[1]:>5.3f} | {elapsed_ms:>5.0f}")

        # ── Esperar hasta el próximo ciclo ───────────────────────────
        # El ESP32 ya marca el ritmo enviando cada 2 s, pero por si acaso:
        dt_used = time.time() - t_step_start
        if dt_used < Ts * 0.9:
            time.sleep(Ts - dt_used)

    print(f"\n  Experimento finalizado — {(time.time()-t_experiment_start)/60:.1f} min")

    return {
        'T_hist'  : T_hist,
        'H_meas'  : H_meas,
        'H2_REAL' : H2_REAL,
        'X_hat'   : X_hat,
        'U_hist'  : U_hist,
        'yref_v'  : yref_v,
        'step_ms' : step_ms,
        'ref_change_times_s': ref_change_times_s,
    }

# ============================================================================
# MÉTRICAS Y GRÁFICAS
# ============================================================================
def compute_metrics(res):
    T    = res['T_hist']
    ref  = np.append(res['yref_v'], res['yref_v'][-1])
    rct  = res['ref_change_times_s']
    idx_400 = T > 400

    settling_mask = np.ones(len(T), dtype=bool)
    for tc in rct:
        settling_mask &= ~((T >= tc) & (T < tc + SETTLING_S))
    mask_ss = idx_400 & settling_mask

    # Errores en cm
    e_h2_ctrl = (res['X_hat'][1, :] - ref) * 100
    e_h1_est  = np.zeros(len(T))   # no tenemos verdad para h1 en la planta real
    e_h3_est  = np.zeros(len(T))

    # Error de estimación de h2 (observador vs verdad terreno — NO usado por el observador)
    e_h2_obs = (res['X_hat'][1, :] - res['H2_REAL']) * 100

    rmse_ctrl   = float(np.sqrt(np.mean(e_h2_ctrl[mask_ss]**2)))
    rmse_h2_obs = float(np.sqrt(np.mean(e_h2_obs[mask_ss]**2)))
    bias_h2_obs = float(np.mean(e_h2_obs[mask_ss]))
    mae_h2_obs  = float(np.mean(np.abs(e_h2_obs[mask_ss])))
    return rmse_ctrl, e_h2_ctrl, mask_ss, e_h2_obs, rmse_h2_obs, bias_h2_obs, mae_h2_obs

def plot_results(res, save_dir=output_dir):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']

    (rmse_ctrl, e_h2_ctrl, mask_ss,
     e_h2_obs, rmse_h2_obs, bias_h2_obs, mae_h2_obs) = compute_metrics(res)
    avg_ms = np.mean(res['step_ms'])
    p99_ms = np.percentile(res['step_ms'], 99)

    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    fig.suptitle(
        f'Koopman Observer+MPC — Real Plant | {datetime.now().strftime("%Y-%m-%d %H:%M")}\n'
        f'RMSE ctrl h₂(SS)={rmse_ctrl:.3f} cm | RMSE obs h₂(SS)={rmse_h2_obs:.3f} cm | '
        f'{avg_ms:.0f} ms/step (p99={p99_ms:.0f} ms)',
        fontsize=10, fontweight='bold')

    # h1 measured
    ax = axes[0, 0]
    ax.plot(T, res['H_meas'][0]*100, 'b', lw=1.5, label='$h_1$ measured')
    ax.plot(T, res['X_hat'][0]*100, 'r--', lw=1.5, label='$h_1$ estimated (Koopman)')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tank 1 (measured)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # h2 estimated + reference + ground truth
    ax = axes[0, 1]
    ax.plot(T, res['X_hat'][1]*100, 'r--', lw=2, label='$h_2$ estimated (Koopman)')
    ax.plot(T, res['H2_REAL']*100, 'g-', lw=1.2, alpha=0.85, label='$h_2$ ground truth')
    ax.stairs(ref[:-1]*100, T, color='k', linestyle='--', lw=1.5, label='Reference')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title(f'Tank 2 (ground truth vs Koopman) — RMSE obs={rmse_h2_obs:.3f} cm')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # h3 measured
    ax = axes[1, 0]
    ax.plot(T, res['H_meas'][1]*100, 'b', lw=1.5, label='$h_3$ measured')
    ax.plot(T, res['X_hat'][2]*100, 'r--', lw=1.5, label='$h_3$ estimated (Koopman)')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tank 3 (measured)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Tracking error
    ax = axes[1, 1]
    ax.plot(T, e_h2_ctrl, 'r', lw=1.5, label=f'RMSE_SS={rmse_ctrl:.3f} cm')
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='excluded transient' if i == 0 else None)
    ax.set_ylabel('Ctrl error $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tracking error'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Control signals
    ax = axes[2, 0]
    ax.stairs(res['U_hist'][0], T, color='b', lw=1.5, label='$u_1$')
    ax.stairs(res['U_hist'][1], T, color='r', lw=1.5, label='$u_3$')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]')
    ax.set_xlabel('t [s]'); ax.set_title('KoopmanMPC inputs')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Computation time
    ax = axes[2, 1]
    ax.plot(res['step_ms'], 'k', lw=0.8, alpha=0.7)
    ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Mean={avg_ms:.0f} ms')
    ax.axhline(p99_ms, color='r', lw=1.5, ls=':', label=f'P99={p99_ms:.0f} ms')
    ax.axhline(Ts * 1000, color='g', lw=1.5, ls='-', label=f'Ts={Ts*1000:.0f} ms')
    ax.set_ylabel('Time [ms]'); ax.set_xlabel('Step k')
    ax.set_title('Computation time per step'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = save_dir / f'koopman_real_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Figura guardada: {fname}")

    # Guardar datos en CSV
    import csv
    csv_path = save_dir / f'datos_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['t_s','h1_meas_m','h2_est_m','h2_real_m','error_obs_cm',
                    'h3_meas_m','ref_m','u1','u3'])
        for k in range(Nsim):
            e_obs_k = (res['X_hat'][1,k] - res['H2_REAL'][k]) * 100
            w.writerow([T[k], res['H_meas'][0,k], res['X_hat'][1,k],
                        res['H2_REAL'][k], e_obs_k,
                        res['H_meas'][1,k], res['yref_v'][k],
                        res['U_hist'][0,k], res['U_hist'][1,k]])
    print(f"  Datos guardados:  {csv_path}")

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 65)
    print("  Koopman Closed-Loop — PLANTA FÍSICA REAL")
    print(f"  Puerto: {SERIAL_PORT} | Ts={Ts}s | T_sim={T_sim}s | {Nsim} pasos")
    print("=" * 65)

    # 1. Calcular puntos estacionarios
    print("\nCalculando puntos de operación nominales...")
    SS_NOM = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS_NOM[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)} | u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → ✗ No convergió (revisa parámetros de orificios)")

    if len(SS_NOM) == 0:
        print("\n  ERROR: No se encontró ningún punto estacionario.")
        print("  Revisa 'alphao', 'Do', 'A2' y 'qi1max'/'qi3max' en P_NOM.")
        sys.exit(1)

    # 2. Entrenar EDMD
    A_d, B_d, nz, nu = train_edmd(verbose=True)

    # 3. Diseñar observador
    L_d, C_obs = design_observer(A_d, nz)

    # 4. Construir MPC
    mpc_args = build_mpc(A_d, B_d, nz, nu)
    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz2, nu2 = mpc_args

    # 5. Conectar al ESP32
    esp = ESP32Interface(port=SERIAL_PORT, baud=BAUD_RATE)
    try:
        esp.connect()

        # ── Verificación antes de empezar ─────────────────────────
        print("\n  Verificación de sensores (5 lecturas):")
        for i in range(5):
            h1, h2_real, h3 = esp.get_measurement()
            print(f"    [{i+1}] h1={h1*100:.2f} cm | h2_real={h2_real*100:.2f} cm | h3={h3*100:.2f} cm")
            time.sleep(0.5)

        input("\n  ¿Todo OK? Presiona ENTER para iniciar el experimento (Ctrl+C para cancelar)...")

        # 6. Lazo de control
        res = run_real_experiment(
            esp, A_d, B_d, L_d, C_obs,
            solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz2, nu2,
            SS_NOM
        )

        # 7. Métricas y gráficas
        rmse_ctrl, _, _, _, rmse_h2_obs, bias_h2_obs, mae_h2_obs = compute_metrics(res)
        print(f"\n  RMSE seguimiento h₂ (estado estacionario): {rmse_ctrl:.4f} cm")
        print(f"  RMSE estimación  h₂ (SS):  {rmse_h2_obs:.4f} cm  ← Koopman vs verdad terreno")
        print(f"  Bias  estimación h₂ (SS):  {bias_h2_obs:.4f} cm")
        print(f"  MAE   estimación h₂ (SS):  {mae_h2_obs:.4f} cm")
        print(f"  Tiempo medio/paso: {np.mean(res['step_ms']):.1f} ms")
        print(f"  Tiempo p99/paso:   {np.percentile(res['step_ms'],99):.1f} ms")
        plot_results(res)

    except KeyboardInterrupt:
        print("\n\n  Interrumpido por el usuario.")
    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback; traceback.print_exc()
    finally:
        esp.close()

if __name__ == '__main__':
    main()

  Koopman Closed-Loop — PLANTA FÍSICA REAL
  Puerto: COM6 | Ts=2.0s | T_sim=800s | 400 pasos

Calculando puntos de operación nominales...
  h2=0.10m → x_ss=[0.1139 0.1    0.0818] | u_ss=[0.3283 0.0012] ✓
  h2=0.15m → x_ss=[0.1675 0.15   0.1269] | u_ss=[0.4029 0.0012] ✓
  h2=0.20m → x_ss=[0.2207 0.2    0.1727] | u_ss=[0.4659 0.0012] ✓

Entrenando EDMD (parámetros nominales)...
  8220 muestras de entrenamiento
  RMSE one-step: 9.74e-05
  |λ| ∈ [0.03444, 0.99990] ✓

Diseñando ganancia L (Luenberger)...
  alpha=1.0: max|λ|=0.94262 ✓

Construyendo KoopmanMPC...
  KoopmanMPC ✓

Conectando a COM6 @ 115200 baud...
  ✓ Conectado a COM6

  Verificación de sensores (5 lecturas):
    [1] h1=1.66 cm | h2_real=1.59 cm | h3=1.35 cm
    [2] h1=1.66 cm | h2_real=1.59 cm | h3=1.35 cm
    [3] h1=1.66 cm | h2_real=1.59 cm | h3=1.35 cm
    [4] h1=1.66 cm | h2_real=1.59 cm | h3=1.35 cm
    [5] h1=1.10 cm | h2_real=1.59 cm | h3=1.35 cm

Esperando primera medición del ESP32...
  Primera medición: h1=1.08cm | 